# DGMAE — Table 6: adaptive discrepancy selection

Reproduce the comparison in §5.10 (DOI: 10.1145/3711896.3736912) on Actor, Chameleon and Squirrel. All four samplers use $p_c=0.2$, $p_\tau=1$, and only the discrepancy reconstruction loss ($\lambda=1$). A full run evaluates 12 settings × 10 seeds. The smoke run checks all 12 settings with one pretraining epoch and two linear-probe epochs.

The notebook reuses the repository's PyTorch model, Table 8 node configuration, Geom-GCN data loader and validation-selected linear probe. The loader symmetrizes and deduplicates edges and standardizes features by column. Each seed uses the corresponding public 48/32/20 split. The paper does not give exact seeds, the degree score formula, or Table 6-specific epoch budgets. This reproduction uses seeds 0–9, one-hop targets (Eq. 9), repository epoch budgets, and the existing degree score $d_i+d_j$, normalized by its mean. Random sampling uses constant $p_c$. The four samplers share $p_c$ and $p_\tau$; their expected sample counts may differ. In particular, attention sampling uses $p_c w_{ij}$ and reverse attention uses $p_c(1-w_{ij})$.

On Kaggle, enable Internet and select a GPU accelerator. Run smoke first. Use `DATASETS` or `METHODS` to split full jobs across notebook sessions if needed.

In [ ]:
from pathlib import Path
import os

MODE = "smoke"  # "smoke" or "full"
DATASETS = ["actor", "chameleon", "squirrel"]
METHODS = ["attn", "inv_attn", "random", "degree"]
SEEDS = [0] if MODE == "smoke" else list(range(10))
DEVICE = "cuda:0"  # Change to "cpu" for a local smoke test.
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
DATA_ROOT = Path(os.environ.get("DGMAE_DATA_ROOT", str(WORK / "dgmae-data")))
OUTPUT = WORK / "dgmae-table6" / MODE

assert MODE in {"smoke", "full"}
assert SEEDS and len(SEEDS) == len(set(SEEDS)) and set(SEEDS) <= set(range(10))

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import json
import platform
import subprocess
import sys

repo_url = "https://github.com/phud774/CS313.R11.git"
revision = "181f21e6356da0119f5789775e83854dbd4aa680"
checkout = next((p for p in (WORK, *WORK.parents) if (p / "DGMAE-REIMPL/train_node.py").is_file()), None)
if checkout is None:
    checkout = WORK / "cs313-table6-code"
    if not (checkout / ".git").exists():
        subprocess.run(["git", "clone", "--no-checkout", repo_url, str(checkout)], check=True)
    subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", revision], check=True)
actual_revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()

try:
    pyg_version = version("torch-geometric")
except PackageNotFoundError:
    pyg_version = None
if pyg_version is None or (WORK == Path("/kaggle/working") and pyg_version != "2.7.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "torch-geometric==2.7.0"], check=True)

source = checkout / "DGMAE-REIMPL"
sys.path.insert(0, str(source))

import pandas as pd
import torch
import torch_geometric
import yaml
from dgmae.data import load_node_dataset
from dgmae.evaluate import linear_probe
from dgmae.utils import set_seed
from train_node import build_model, pretrain

device = torch.device(DEVICE)
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU or set DEVICE to cpu for smoke testing.")
environment = {
    "commit": actual_revision, "paper_doi": "10.1145/3711896.3736912",
    "python": platform.python_version(), "torch": torch.__version__,
    "pyg": torch_geometric.__version__, "device": str(device),
    "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
}
print(environment)

In [ ]:
import statistics

paper = {
    "actor": {"attn": (34.34, 0.41), "inv_attn": (35.36, 0.68), "random": (34.72, 0.74), "degree": (35.12, 0.64)},
    "chameleon": {"attn": (70.88, 1.09), "inv_attn": (73.90, 1.92), "random": (71.80, 2.13), "degree": (73.44, 1.53)},
    "squirrel": {"attn": (58.20, 1.98), "inv_attn": (65.61, 1.35), "random": (60.07, 1.67), "degree": (60.63, 1.16)},
}
assert DATASETS and len(DATASETS) == len(set(DATASETS)) and set(DATASETS) <= set(paper)
assert METHODS and len(METHODS) == len(set(METHODS)) and set(METHODS) <= set(paper["actor"])
node_config = yaml.safe_load((source / "configs/node.yml").read_text())
OUTPUT.mkdir(parents=True, exist_ok=True)
scores = []
summary = []

def write_json(path, payload):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n")
    temp.replace(path)

for dataset in DATASETS:
    x, y, graph, masks, info = load_node_dataset(dataset, root=str(DATA_ROOT), device=device)
    assert info["num_splits"] == 10, info
    print(info, flush=True)
    for method in METHODS:
        cfg = dict(node_config[dataset], selection=method, lam=1.0, p_c=0.2, p_tau=1.0, hop=1)
        if MODE == "smoke":
            cfg["max_epoch"] = 1
        probe_epochs = 2 if MODE == "smoke" else 300
        directory = OUTPUT / dataset / method
        directory.mkdir(parents=True, exist_ok=True)
        settings = {"dataset": dataset, "method": method, "mode": MODE, "environment": environment,
                    "data": info, "config": cfg, "probe_epochs": probe_epochs}
        settings_path = directory / "settings.json"
        if settings_path.exists() and json.loads(settings_path.read_text()) != settings:
            raise ValueError(f"Existing settings differ. Choose another OUTPUT directory: {directory}")
        write_json(settings_path, settings)
        trial = []
        for seed in SEEDS:
            path = directory / f"seed-{seed}.json"
            if path.is_file():
                record = json.loads(path.read_text())
            else:
                print(f"{dataset} / {method} / seed {seed}", flush=True)
                set_seed(seed)
                model = build_model(x.size(1), cfg).to(device)
                history = pretrain(model, x, graph, cfg, log_every=100)
                model.eval()
                embedding = model.embed(x, graph)
                train, val, test = (mask[:, seed:seed + 1] for mask in masks)
                set_seed(seed)
                accuracy, = linear_probe(embedding, y, train, val, test, info["num_classes"], epochs=probe_epochs)
                record = dict(dataset=dataset, method=method, seed=seed, split=seed,
                              accuracy_pct=accuracy, final_training_loss=history[-1]["loss"])
                write_json(path, record)
                del model, embedding, history
            trial.append(record["accuracy_pct"])
            scores.append(record)
        row = dict(dataset=dataset, method=method, runs=len(trial), mode=MODE,
                   accuracy_mean_pct=statistics.mean(trial), accuracy_sd_pct=statistics.pstdev(trial))
        if MODE == "full":
            row["paper_mean_pct"], row["paper_sd_pct"] = paper[dataset][method]
        summary.append(row)
        pd.DataFrame(scores).to_csv(OUTPUT / "per_seed.csv", index=False)
        pd.DataFrame(summary).to_csv(OUTPUT / "table6.csv", index=False)

pd.DataFrame(summary).round(2)